In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BorsaBot — 03 | Triple Barrier Labeling (Etiketleme)
# ═══════════════════════════════════════════════════════════════════
# ohlcv, features, CFG mevcut olmalı.

In [ ]:
# ─────────────────────────────────────────────────────────────────
def compute_daily_vol(close: pd.Series, span: int = 100) -> pd.Series:
    """EWM volatilite. Triple Barrier bariyer genişliği için."""
    ret = close.pct_change().dropna()
    return ret.ewm(span=span, min_periods=span).std()

In [ ]:
# ─────────────────────────────────────────────────────────────────
def get_events(close: pd.Series,
               t_events: pd.DatetimeIndex,
               pt_sl: list[float],
               trgt: pd.Series,
               vertical_bars: int = 24,  # max holding period (bar sayısı)
               min_ret: float = 0.0) -> pd.DataFrame:
    """Her olay için (t0 → t1, hedef volatilite, yön) tablosu."""
    trgt = trgt.reindex(t_events).dropna()
    if min_ret > 0:
        trgt = trgt[trgt > min_ret]

    # Dikey bariyer: t0'dan vertical_bars bar sonra
    t1_list = []
    for t in trgt.index:
        loc = close.index.searchsorted(t)
        end_loc = min(loc + vertical_bars, len(close) - 1)
        t1_list.append(close.index[end_loc])

    t1 = pd.Series(t1_list, index=trgt.index)
    return pd.DataFrame({"t1": t1, "trgt": trgt, "side": 1.0})

In [ ]:
# ─────────────────────────────────────────────────────────────────
def triple_barrier_labels(close: pd.Series,
                          events: pd.DataFrame,
                          pt_sl: list[float] = (1.0, 1.0)) -> pd.Series:
    """
    +1  → profit-take bariyeri önce kırıldı
    -1  → stop-loss bariyeri önce kırıldı
     0  → süre doldu (dikey bariyer)
    """
    labels = {}
    for t0, row in events.iterrows():
        t1, trgt, side = row["t1"], row["trgt"], row.get("side", 1.0)
        if t0 not in close.index:
            continue
        path = close.loc[t0:t1]
        if path.empty:
            labels[t0] = 0
            continue

        ret = (path / close.loc[t0]) - 1.0
        pt  =  trgt * pt_sl[0] * side
        sl  = -trgt * pt_sl[1] * abs(side)

        up_dates   = ret[ret >=  pt].index
        down_dates = ret[ret <=  sl].index

        up_t   = up_dates[0]   if len(up_dates)   > 0 else pd.NaT
        down_t = down_dates[0] if len(down_dates) > 0 else pd.NaT

        if pd.isna(up_t) and pd.isna(down_t):
            labels[t0] = 0
        elif pd.isna(down_t) or (not pd.isna(up_t) and up_t <= down_t):
            labels[t0] = 1
        else:
            labels[t0] = -1

    return pd.Series(labels, name="label")

In [ ]:
# ─────────────────────────────────────────────────────────────────
VERTICAL_BARS = 48    # max 48 saat tutma
PT_SL         = [1.5, 1.0]   # profit-take:1.5x vol, stop:1.0x vol
MIN_RET       = 0.0005        # min %0.05 getiri eşiği

labeled: dict[str, pd.DataFrame] = {}

for sym in CFG["symbols"]:
    close = ohlcv[sym]["close"]
    feat  = features[sym]

    # CUSUM filtresi: sadece volatil anlarda sinyal üret
    vol   = compute_daily_vol(close)
    trgt  = vol.reindex(feat.index).dropna()

    # Tüm feature indexini kullan (CUSUM yerine sade yaklaşım)
    t_events = trgt.index

    events = get_events(
        close    = close,
        t_events = t_events,
        pt_sl    = PT_SL,
        trgt     = trgt,
        vertical_bars = VERTICAL_BARS,
        min_ret  = MIN_RET,
    )

    labels = triple_barrier_labels(close, events, pt_sl=PT_SL)

    # Birleştir
    df_labeled = feat.loc[labels.index].copy()
    df_labeled["label"]  = labels
    df_labeled["t1"]     = events["t1"].reindex(labels.index)
    df_labeled["trgt"]   = events["trgt"].reindex(labels.index)
    df_labeled = df_labeled.dropna(subset=["label"])

    labeled[sym] = df_labeled
    df_labeled.to_parquet(f"{CFG['out_dir']}/{sym}_labeled.parquet")

    dist = df_labeled["label"].value_counts().sort_index()
    print(f"\n[{sym}] {len(df_labeled):,} etiket")
    print(f"  -1 (SELL): {dist.get(-1, 0):5,}  ({dist.get(-1,0)/len(df_labeled)*100:.1f}%)")
    print(f"   0 (FLAT): {dist.get( 0, 0):5,}  ({dist.get( 0,0)/len(df_labeled)*100:.1f}%)")
    print(f"  +1 (BUY) : {dist.get( 1, 0):5,}  ({dist.get( 1,0)/len(df_labeled)*100:.1f}%)")

print("\n✓ Etiketleme tamamlandı")